# PAI 最終課題 kojikoji_s
## オフィスなどにおけるノート型パソコン搬送に向けた双腕ロボットの基礎検証
搬送物は、ノート型パソコンを想定したケース形状の簡略化モデルです。固定環境内の受取台への搬送を検証し、移動台車や人への直接手渡しは含みません。
ROS 2 / Gazebo / LeRobot $\pi_{0.5}$。実行順は環境構築 → データ収集 → 学習 → 評価です。全セル実行をしないでください。Google ColabのA100 GPU（GPUメモリ80GB）を使用し、ハイメモリ設定を選択します。GPUメモリとシステムRAMは別の資源です。既に成功動画がある場合は再実行不要です。
2026年10月18日、現行公開版のUAT（ユーザー受入テスト）が完了したことを利用者が確認しました。7.2に保存済み記録を読み取る可視化セルを追加しました。既存の学習・推論・制御セルは変更していません。追加セルは合成データで読み込み・グラフ・動画出力を確認していますが、Colabの実データでの表示確認は別途必要です。今後プログラムや依存環境を変更した場合は、その変更に応じて再検証してください。

## 学習・推論を行う箇所

**学習はノートブックの4・5、推論は6で行います。いずれもColabのGPUを使用します。**

| 段階 | 主なプログラム | 処理 |
|---|---|---|
| 3 教師データ収集 | `it_desk/collect.py` | 設計済みの教師制御でGazeboのロボットを動かし、画像・関節状態・正解動作を収集。この段階ではモデルを学習しません。 |
| 4 初回学習 | `it_desk/train.py` | 成功した教師データを用い、事前学習済み$\pi_{0.5}$の動作生成部分を500ステップ追加学習します。 |
| 5 追加学習 | `it_desk/train_refine.py` | 4の重みを引き継ぎ、さらに1,000ステップ学習します。 |
| 6 評価・推論 | `scripts/evaluate.py` → `it_desk/policy.py` | 学習済み重みを読み込み、2視点の画像・関節状態・タスク指示から、次の関節目標の列を予測します。評価中は重みを更新しません。 |

追加学習した重みは、実行フォルダ内の`checkpoint_refined/expert.safetensors`に保存され、6の推論時に読み込まれます。

**タスク固有データで追加学習した$\pi_{0.5}$方策が予測した関節目標を、そのままロボットへ送る構成ではありません。** `it_desk/run_hybrid.py`が評価試行を設定し、`it_desk/episode.py`が制御ループを実行します。その中で`it_desk/cooperative.py`が学習済み方策の予測を設計済み経路の方向へ射影し、経路上の進行量に反映します。経路の形状・工程順序・双腕の同期・関節制限は通常の制御が担当します。推論待ちの間はGazeboの物理シミュレーションの進行を一時停止します。



In [ ]:
from pathlib import Path
import os, sys, subprocess, json, datetime, getpass, hashlib, shutil
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('Google ColabのLinux GPUランタイムを使用してください。')
ROOT=Path('/content/pai-laptop-handover')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/1976koji/pai-laptop-handover.git',str(ROOT)],check=True)
RUN=Path('/content/pai_runs')/datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN.mkdir(parents=True,exist_ok=False)
AI_PYTHON=Path('/content/pai_kojikoji_s/venv_cu128_v4/bin/python')
def run(args):
    env=os.environ.copy()
    for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
        env.pop(key,None)
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True,env=env)
print('RUN:',RUN)
run(['nvidia-smi'])


## 1 環境構築
APT導入には時間がかかります。ロボット資産は固定revisionから取得します。

In [ ]:
run(['bash','it_desk/setup_sim.sh'])
run([sys.executable,'it_desk/setup_ai.py','--venv',AI_PYTHON.parent.parent])
run(['bash','scripts/ros_run.sh','prepare_robot.py'])
run(['bash','scripts/ros_run.sh','verify_static.py'])
run(['bash','scripts/ros_run.sh','test_control_interface.py'])


## 2 Hugging Face認証処理の定義

このセルはそのまま実行してください。ここでのトークン入力は不要です。

学習・評価時に、利用者本人のColab Secretsから`HF_TOKEN`を取得します。取得できない場合のみ、非表示の入力欄が表示されます。処理が成功したトークンは同じセッションのメモリ内で再利用し、ファイルやカーネルの環境変数には保存しません。セッションの終了・再起動、またはこのセルの再実行で保持は解除されます。


In [ ]:
# 同一セッション内だけで、成功した処理のトークンを再利用します。
for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
    os.environ.pop(key,None)
_hf_token_cache=None

def clear_hf_token():
    global _hf_token_cache
    _hf_token_cache=None

def require_trained(folder, steps):
    folder=Path(folder)
    report_path=folder/'training_result.json'
    weights=folder/'expert.safetensors'
    if not report_path.is_file() or not weights.is_file():
        raise RuntimeError('学習完了ファイルがありません。前の学習セルを完了してください。')
    report=json.loads(report_path.read_text())
    if (report.get('status')!='trained'
        or report.get('task_training_completed') is not True
        or report.get('task')!='it_laptop_handover_fixed_cell'
        or report.get('steps_completed',0)<steps
        or weights.stat().st_size==0):
        raise RuntimeError('前の学習が未完了または失敗しています。トークン入力と次の処理は行いません。')
    return report

def run_ai(args, completed_check=None):
    global _hf_token_cache
    env=os.environ.copy()
    for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
        env.pop(key,None)
    token=None
    try:
        token=_hf_token_cache
        if not token:
            try:
                from google.colab import userdata
                token=userdata.get('HF_TOKEN')
            except Exception:
                token=None
            if not token:
                token=getpass.getpass('Hugging Face読み取りトークン（非表示）: ')
        if not token:raise RuntimeError('トークンが空です。')
        env['HF_TOKEN']=token
        subprocess.run([str(x) for x in args],cwd=ROOT,check=True,env=env)
        if completed_check is not None:
            completed_check()
        _hf_token_cache=token
    except BaseException:
        clear_hf_token()
        raise
    finally:
        env.pop('HF_TOKEN',None)
        token=None

def train_stage(folder, steps, args):
    # 完了済みなら再学習しません。不完全な既存重みは上書きしません。
    folder=Path(folder)
    if (folder/'expert.safetensors').exists():
        require_trained(folder,steps)
        print('学習完了済み。既存モデルを使用します。')
        return
    run_ai(args,completed_check=lambda:require_trained(folder,steps))
    print('学習完了を確認しました。次のセルへ進めます。')


## 3 教師データ収集
失敗を含めて保存します。

`it_desk/collect.py`で教師制御の実行データを収集します。この段階ではモデルの重みは更新しません。


In [ ]:
run(['bash','scripts/ros_run.sh','collect.py','--out',RUN/'train','--count','24','--seed','100','--duration','150'])
run(['bash','scripts/ros_run.sh','collect.py','--out',RUN/'validation','--count','6','--seed','300','--duration','150'])
for split,minimum in [('train',12),('validation',4)]:
    s=json.loads((RUN/split/'summary.json').read_text())
    print(split,'usable:',s['usable_successes'])
    if s['usable_successes']<minimum:raise RuntimeError('成功データ不足。失敗の原因を確認してください。')


## 4 初回学習 500ステップ
3の成功データ数を確認してから実行してください。

`it_desk/train.py`で、教師データから$\pi_{0.5}$の動作生成部分を500ステップ追加学習します。重みは`RUN/checkpoint/expert.safetensors`に保存します。


In [ ]:
train_stage(RUN/'checkpoint',500,
    [AI_PYTHON,'it_desk/train.py','--train',RUN/'train','--validation',RUN/'validation','--out',RUN/'checkpoint','--steps','500'])


## 5 追加学習 1000ステップ

初回学習の完了を確認してから実行します。失敗・未完了の場合はトークンを求めず停止します。同じセッションでは初回学習で成功したトークンを再利用します。

`it_desk/train_refine.py`で初回学習の重みを引き継ぎ、さらに1,000ステップ学習します。重みは`RUN/checkpoint_refined/expert.safetensors`に保存します。


In [ ]:
require_trained(RUN/'checkpoint',500)
train_stage(RUN/'checkpoint_refined',1000,
    [AI_PYTHON,'it_desk/train_refine.py','--train',RUN/'train','--validation',RUN/'validation','--init',RUN/'checkpoint','--out',RUN/'checkpoint_refined','--steps','1000'])
CHECKPOINT=RUN/'checkpoint_refined'


## 6 学習済み方策と双腕協調制御で評価
評価条件は[`it_desk/run_hybrid.py`](https://github.com/1976koji/pai-laptop-handover/blob/main/it_desk/run_hybrid.py)の`main()`内の`trials`で指定しています。`seed=900`は`developer`、`seed=901`は`standard`に対応します。ノートブックの6から`scripts/evaluate.py`を通じてこの2試行を実行し、結果は評価フォルダ内の`evaluation/summary.json`に記録します。これらは開発時にも使用した条件であり、未知の配置での性能は未検証です。

`scripts/evaluate.py`が`it_desk/policy.py`を起動し、5で保存した重みを使って推論します。画像・関節状態・指示から関節目標を予測し、双腕協調制御で経路上の進行量へ変換します。この段階では学習しません。


In [ ]:
CHECKPOINT=RUN/'checkpoint_refined'
require_trained(CHECKPOINT,1000)
EVAL=RUN/('evaluation_'+datetime.datetime.now().strftime('%H%M%S'))
run_ai([sys.executable,'scripts/evaluate.py','--python',AI_PYTHON,'--checkpoint',CHECKPOINT,'--out',EVAL])


## 7 再評価動画の確認

今回の評価から`EVAL/review_2x.mp4`を生成します。画面に結果と2倍速表記を表示します。失敗試行も含まれるため、提出前に評価結果と映像を確認してください。長さの要件判定は出力されますが、範囲外でも自動停止はしません。1〜3分以内であることを確認してください。

**Driveの保存済みデータを使う場合は、この7の動画生成セルを実行せず、7.1 → 7.2 → 8の順に進んでください。**


In [ ]:
run([sys.executable,'scripts/make_review.py','--evaluation',EVAL/'evaluation','--out',EVAL/'review_2x.mp4'])
from IPython.display import Video,display
display(Video(str(EVAL/'review_2x.mp4'),embed=True))


## 7.1 保存済みデータから可視化する場合（任意）

**新しいセッションでDrive上の記録を使うときは、この7.1のセル → 7.2 → 8の順に実行してください。1〜6の再実行・再学習は不要です。** 現在のセッションで評価を終えた直後なら、このセルを省略して7.2へ進めます。

通常の保存先は`MyDrive/PAI_results/実行日時/`です。下のセルは実行日時（`YYYYMMDD_HHMMSS`）が新しい順に確認し、画像・制御記録がある最初のフォルダを自動選択します。最新の実行が未保存・空・モデルのみの場合は除外します。成功試行だけを選ぶ処理ではなく、失敗試行も対象です。

`PAI_backup`は以前の別の保存先であり、両方は必要ありません。そこに保存した記録を使う場合だけ、`SAVED_ROOT`を変更してください。日時は実行フォルダ作成日時であり、プログラムのバージョン番号や学習の完了日時ではありません。

実行フォルダの下には`train/episode_000/`、`validation/episode_000/`、`evaluation_時刻/evaluation/hybrid_900/`などがあります。下のセルで実行フォルダを選び、7.2の「試行」で個々の記録を選べます。7.2ではパス名を降順に並べ、`hybrid_900`を優先します。評価名の時刻が日付を含まない場合、日をまたぐ再評価の新旧は名前だけでは確定できないため、表示された試行パスを確認してください。


In [ ]:
# 保存済みデータを使う場合だけ実行。学習・評価は再実行しません。
from pathlib import Path
import datetime, json, re
from google.colab import drive

drive.mount('/content/drive')
# 通常はこの1か所で十分です。以前のバックアップを使う場合のみ
# '/content/drive/MyDrive/PAI_backup' などへ変更してください。
SAVED_ROOT = Path('/content/drive/MyDrive/PAI_results')

def saved_run_date(folder):
    match = re.match(r'^(\d{8}_\d{6})(?:$|_)', folder.name)
    if not match:
        return None
    try:
        return datetime.datetime.strptime(match.group(1), '%Y%m%d_%H%M%S')
    except ValueError:
        return None

def has_visual_records(folder):
    # モデルファイルだけの保存先は選びません。画像・制御記録を確認します。
    for trace_path in folder.rglob('control_trace.json'):
        episode = trace_path.parent
        try:
            trace = json.loads(trace_path.read_text())
            if not isinstance(trace, list) or len(trace) < 2:
                continue
            if not all(isinstance(r,dict) and all(k in r for k in ('sim_t','q','command','case')) for r in trace):
                continue
            result_path = episode/'result.json'
            result = json.loads(result_path.read_text()) if result_path.is_file() else {}
            if result.get('mode') == 'teacher' and (episode/'episode.npz').is_file():
                return True
            index_path = episode/'frames.json'
            if index_path.is_file():
                index = json.loads(index_path.read_text())
                count = sum((episode/'frames'/f'{int(row[1]):05d}.jpg').is_file() for row in index)
                if count >= 2:
                    return True
        except (OSError, ValueError, TypeError, KeyError, IndexError):
            continue
    return False

def choose_latest_saved_run(root):
    root = Path(root)
    if not root.is_dir():
        raise RuntimeError('保存先がありません。SAVED_ROOTを実際の保存先に変更してください: '+str(root))
    # 実行フォルダ自体を指定した場合も受け付けます。
    if saved_run_date(root) is not None:
        candidates = [root]
    else:
        candidates = [p for p in root.iterdir() if p.is_dir() and saved_run_date(p) is not None]
    candidates.sort(key=lambda p:(saved_run_date(p),p.name),reverse=True)
    for folder in candidates:
        print('記録を確認:',folder.name,flush=True)
        if has_visual_records(folder):
            return folder
        print('  画像・制御記録が不足しているため、このフォルダは除外します。')
    raise RuntimeError('日時付き実行フォルダに可視化用記録が見つかりません。保存先を確認してください。再学習は開始しません。')

# 前の選択を誤って再利用しないよう解除してから検索します。
VIS_RUN = None
VIS_RUN = choose_latest_saved_run(SAVED_ROOT)
print('\n自動選択した実行フォルダ:', VIS_RUN)
print('次は7.2だけ実行してください。別の記録を使う場合はVIS_RUN = Path("実行フォルダのパス")で変更できます。')


## 7.2 カメラ画像とロボット挙動の可視化（任意・再学習不要）

保存済み記録のみを読み込みます。ROS・方策サーバーを起動せず、既存の重み・評価結果を変更しません。既に結果がある場合は、**このセルだけ**実行できます。再接続した場合は、先に7.1で保存済みフォルダを自動選択してください。新しい初期化セルを実行して空のRUNを作る必要はありません。

- 試行と表示する関節を選び、「記録を読み込み・グラフ保存」を押します。失敗試行も選択できます。
- **教師試行**：`episode.npz`のfrontとoverheadを横並び表示します。これは教師制御の画像であり、学習済み方策の評価映像ではありません。既存データのキー`top`はfront、`side`はoverheadを指します。
- **方策評価試行**：保存されたfront映像を表示します。現在の評価コードはoverhead画像を保存しないため、「未記録」と表示します。教師試行の画像で代用したり、上方画像を再生成したりしません。frontは提出動画と同じ視点ですが、保存JPEGには字幕・圧縮が含まれ、方策への生の入力画像とは同一ではありません。
- グラフは①画像の平均絶対差分、②方策由来の進行係数α、③選択関節の予測目標・制御指令・観測角度、④観測関節角度の変化速度のノルム、⑤搬送物と左右支持板の高さを**シミュレーション時間**で表示します。画像番号のスライダーで画像と最近傍の制御時刻・工程を確認できます。赤い縦線が選択画像の時刻を示します。
- 画像差分は、隣接する保存画像の各RGB成分の差の絶対値を平均して255で割った**後処理の指標**です。評価JPEGでは字幕領域（上端50行）を除きます。モデル内部の画像特徴や注意重みではなく、画像による因果的寄与を示すものでもありません。搬送物の高さはシミュレータ、支持板位置は順運動学から得た値で、画像認識による位置推定値ではありません。
- 各カメラは最新画像を非同期に取得しており、厳密な同時撮影は保証されません。評価記録から各推論に実際に渡した生画像を完全に対応づけることはできません。グラフは観測された時間的な対応を確認するためのものです。

PNG・CSVは`RUN/visualization/日時/試行名/`に保存されます。「カメラ映像MP4を保存」で映像も出力します。映像は5 fpsのシミュレーション時間軸へ直前画像を保持して再標本化した**1倍速の補助映像**です。提出動画`review_2x.mp4`とは別で、課題の動画尺を満たすとは限りません。カメラ画像2枠と5グラフを1枚の画面（標準幅900ピクセル以内）に表示します。「拡大表示」で細部を確認し、「表示画像PNGをダウンロード」で保存できます。カメラ画像はfront_selected.pngとoverhead_selected.pngにも保存します（未記録の視点は出力しません）。補助MP4の出力欄は折り畳み表示です。必要な試行を可視化した後、8でDriveへの保存を確認してください。Drive上の記録を直接使用した場合、可視化結果もその実行フォルダ内へ直接保存されています。


In [ ]:
# 読み取り専用の後処理。学習・評価・ROS制御は実行しません。
from pathlib import Path
import json, datetime, csv, subprocess, shutil, io, base64, html
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display, clear_output, Video

# 再接続後は、保存済みの実行フォルダを明示してください（自動で最新を選びません）。
VIS_RUN = (Path(VIS_RUN) if globals().get('VIS_RUN') is not None
           else Path(RUN) if 'RUN' in globals() else None)
if VIS_RUN is None or not VIS_RUN.is_dir():
    raise RuntimeError('先に7.1で保存済みデータを選択してください。再学習は不要です。')
episodes = sorted({p.parent for p in VIS_RUN.rglob('control_trace.json')}, reverse=True)
if not episodes:
    raise RuntimeError('control_trace.jsonがありません。評価・教師試行の保存先を確認してください。')
VIS_OUT = VIS_RUN/'visualization'/datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
VIS_OUT.mkdir(parents=True, exist_ok=False)
print('可視化出力:', VIS_OUT)
print('時刻はシミュレーション時間。画像差分は後処理の参考指標で、モデル内部特徴や因果的寄与ではありません。')

def load_visual_episode(folder):
    trace = json.loads((folder/'control_trace.json').read_text())
    if len(trace) < 2:
        raise ValueError('観測記録が2点未満です。')
    t = np.array([r['sim_t'] for r in trace], dtype=float)
    if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('制御時刻が有限かつ単調増加ではありません。')
    result = json.loads((folder/'result.json').read_text()) if (folder/'result.json').exists() else {}
    data = dict(folder=folder, trace=trace, t=t, result=result)
    npz = folder/'episode.npz'
    if npz.is_file() and result.get('mode') == 'teacher':
        with np.load(npz, allow_pickle=False) as a:
            data.update(it=a['time'].copy(), front=a['top'].copy(), overhead=a['side'].copy())
        # 既存のキー名に注意: top=front、side=overhead（episode.pyの格納順）。
        data['kind'] = 'teacher: stored 224x224 RGB observations'
        data['frame'] = lambda i: (data['front'][i], data['overhead'][i])
    else:
        index = json.loads((folder/'frames.json').read_text())
        valid = [(float(ti), folder/'frames'/f'{int(fi):05d}.jpg') for ti,fi in index
                 if (folder/'frames'/f'{int(fi):05d}.jpg').is_file()]
        if len(valid) < 2:
            raise ValueError('保存された斜め前方の画像が2枚未満です。')
        data['it'] = np.array([v[0] for v in valid]); data['paths'] = [v[1] for v in valid]
        def frame(i):
            with Image.open(data['paths'][i]) as im:
                return np.array(im.convert('RGB')), None
        data['frame'] = frame
        data['kind'] = 'recorded front JPEG; overhead not saved'
    it = np.asarray(data['it'], dtype=float)
    if len(it) < 2 or not np.isfinite(it).all() or np.any(np.diff(it) <= 0):
        raise ValueError('画像時刻が不足、重複、または非単調です。')
    diff = np.full((len(it), 2), np.nan)
    previous = [None, None]
    for i in range(len(it)):
        for j, image in enumerate(data['frame'](i)):
            if image is None: continue
            # 評価映像は説明字幕の行を除外。生の方策入力画像とは同一ではありません。
            if 'paths' in data: image = image[50:, :, :]
            pixels = np.asarray(Image.fromarray(image).resize((224,224)), dtype=np.float32)
            if previous[j] is not None: diff[i,j] = np.mean(np.abs(pixels-previous[j]))/255.
            previous[j] = pixels
    data['image_change'] = diff
    data['q'] = np.array([r['q'] for r in trace], dtype=float)
    data['cmd'] = np.array([r['command'] for r in trace], dtype=float)
    if data['q'].shape != (len(t),14) or data['cmd'].shape != (len(t),14):
        raise ValueError('14関節の記録形式ではありません。')
    data['pred'] = np.array([(r.get('projection') or {}).get('learned_joint_goal',[np.nan]*14) for r in trace])
    data['alpha'] = np.array([(r.get('projection') or {}).get('progress_fraction',np.nan) for r in trace])
    data['speed'] = np.r_[np.nan, np.linalg.norm(np.diff(data['q'],axis=0),axis=1)/np.diff(t)]
    return data

def save_tables(data, out):
    out.mkdir(parents=True, exist_ok=True)
    # 画像と制御は元の時刻のまま別CSVへ保存。補間値を実測値と混同しない。
    with (out/'image_metrics.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['image_sim_t_s','front_mean_abs_pixel_change_0_1','overhead_mean_abs_pixel_change_0_1'])
        w.writerows(zip(data['it'],*data['image_change'].T))
    with (out/'control_metrics.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['control_sim_t_s','phase','alpha','joint_speed_norm_rad_per_s','case_x_m','case_y_m','case_z_m']+
            [f'{kind}_{side}{i}_rad' for kind in ['observed','command','learned_goal'] for side in ['L','R'] for i in range(1,8)])
        for i,r in enumerate(data['trace']):
            w.writerow([data['t'][i],r['phase'],data['alpha'][i],data['speed'][i],*r['case'],*data['q'][i],*data['cmd'][i],*data['pred'][i]])
    (out/'README.txt').write_text(
        'Postprocessing only; not a new robot evaluation.\n'+data['kind']+'\n'+
        'Camera snapshots are asynchronous latest observations, not hardware-synchronized frames.\n'
        'Pixel change = mean(abs(current-previous))/255; not learned features or causal attribution.\n'
        'Recorded front JPEG metrics exclude the top 50 caption rows; they are not raw policy inputs.\n'
        'Case/support positions come from simulator state/kinematics, not image recognition.\n'
        'Command and learned_goal are targets, not measured motion. Teacher trials have no policy alpha.\n'
        'Video uses a uniform 5 fps simulation-time grid with latest available frames (sample-and-hold).\n'
        'No timestamps for individual raw policy input images are available in evaluation logs.\n',encoding='utf-8')

def make_plot(data, joint, image_index=0):
    # カメラ2視点と5グラフを単一の画面に配置する。
    t=data['t'];tr=data['trace'];stamp=float(data['it'][image_index])
    fig=plt.figure(figsize=(12,8),layout='constrained')
    grid=fig.add_gridspec(4,2,height_ratios=[1.5,1,1,1])
    for j,(name,image) in enumerate(zip(['front','overhead'],data['frame'](image_index))):
        ax=fig.add_subplot(grid[0,j]);ax.set_axis_off()
        if image is None:
            ax.text(.5,.5,'OVERHEAD NOT RECORDED\nThis evaluation saved front only',ha='center',va='center',transform=ax.transAxes)
        else:ax.imshow(image)
        ax.set_title(name+' | sim t='+f'{stamp:.3f}s',fontsize=10)
    axes=[fig.add_subplot(grid[1,0]),fig.add_subplot(grid[1,1]),fig.add_subplot(grid[2,0]),fig.add_subplot(grid[2,1]),fig.add_subplot(grid[3,:])]
    for j,name in enumerate(['front','overhead']):
        if np.isfinite(data['image_change'][:,j]).any():axes[0].plot(data['it'],data['image_change'][:,j],label=name)
    axes[0].set_ylabel('Pixel change [0-1]');axes[0].legend(fontsize=7)
    if np.isfinite(data['alpha']).any():axes[1].plot(t,data['alpha'])
    else:axes[1].text(.05,.5,'Teacher: no policy alpha',transform=axes[1].transAxes,fontsize=9)
    axes[1].set_ylabel('Progress alpha');axes[1].set_ylim(-.05,1.05)
    for key,label in [('q','observed'),('cmd','command'),('pred','learned goal')]:
        if np.isfinite(data[key][:,joint]).any():axes[2].plot(t,data[key][:,joint],label=label)
    axes[2].set_ylabel(f"{'L' if joint<7 else 'R'}{joint%7+1} [rad]");axes[2].legend(fontsize=7,ncol=3)
    axes[3].plot(t,data['speed']);axes[3].set_ylabel('Joint speed [rad/s]');axes[3].set_ylim(0,max(.01,float(np.nanmax(data['speed']))*1.15))
    for key,label in [('case','laptop (simulator)'),('left_support','left support (FK)'),('right_support','right support (FK)')]:
        axes[4].plot(t,[r[key][2] for r in tr],label=label)
    axes[4].set_ylabel('Height [m]');axes[4].legend(fontsize=7,ncol=3)
    for ax in axes:
        ax.axvline(stamp,color='crimson',linestyle='--',linewidth=1)
        ax.set_xlabel('Simulation time [s]',fontsize=8);ax.tick_params(labelsize=8);ax.grid(alpha=.2)
    fig.suptitle(f"{data['folder'].name} | mode={data['result'].get('mode','unknown')} | success={data['result'].get('success','unknown')}",fontsize=11)
    return fig,axes

def camera_panel(data, i):
    canvas=Image.new('RGB',(900,320),(20,25,35));draw=ImageDraw.Draw(canvas)
    for k,(name,im) in enumerate(zip(['front','overhead'],data['frame'](i))):
        draw.text((15+450*k,10),name,fill='white')
        if im is None:draw.text((465,130),'Not recorded for this evaluation',fill='white')
        else:
            view=Image.fromarray(im);view.thumbnail((430,245));canvas.paste(view,(15+450*k,35))
    label=f"mode={data['result'].get('mode','unknown')} success={data['result'].get('success','unknown')} | sim t={data['it'][i]:.3f}s"
    draw.text((15,285),label,fill='white');draw.text((15,303),'Camera record / 1x simulation time (resampled at 5 fps)',fill='white')
    return canvas

def export_camera_video(data, out):
    ffmpeg=shutil.which('ffmpeg')
    if not ffmpeg: raise RuntimeError('ffmpegがありません。環境構築済みのColabで実行してください。')
    target=out/'camera_views_1x.mp4';tmp=out/'camera_views_build.mp4'
    times=np.arange(data['it'][0],data['it'][-1]+1e-8,.2)
    log=out/'camera_encode.log'
    with log.open('w') as err:
        p=subprocess.Popen([ffmpeg,'-y','-loglevel','error','-f','rawvideo','-pix_fmt','rgb24','-s','900x320','-r','5','-i','-',
            '-an','-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(tmp)],stdin=subprocess.PIPE,stderr=err)
        try:
            for t in times:
                i=int(np.clip(np.searchsorted(data['it'],t,side='right')-1,0,len(data['it'])-1))
                p.stdin.write(np.asarray(camera_panel(data,i),dtype=np.uint8).tobytes())
            p.stdin.close();rc=p.wait()
            if rc:raise RuntimeError('映像出力失敗: '+str(log))
        finally:
            if p.poll() is None:p.terminate();p.wait()
    tmp.replace(target);return target

# 一試行ずつ選択。異なる試行の画像と動作を混ぜません。
options=[(str(p.relative_to(VIS_RUN)),p) for p in episodes]
default=next((p for p in episodes if p.name=='hybrid_900'),episodes[0])
choice=widgets.Dropdown(options=options,value=default,description='試行:',layout=widgets.Layout(width='90%'))
joint_choice=widgets.Dropdown(options=[(f'{s}{i+1}',j*7+i) for j,s in enumerate(['L','R']) for i in range(7)],description='関節:')
load_button=widgets.Button(description='記録を読み込み・グラフ保存')
output=widgets.Output();state={}

def show_episode(_):
    load_button.disabled=True
    with output:
        clear_output(wait=True)
        try:
            data=load_visual_episode(choice.value)
            out=VIS_OUT/('_'.join(choice.value.relative_to(VIS_RUN).parts))
            save_tables(data,out);state.clear();state.update(data=data,out=out)
            print(data['kind']);print('保存先:',out)
            print('赤線＝選択画像の時刻。未記録の上方画像は補完しません。')
            slider=widgets.IntSlider(min=0,max=len(data['it'])-1,value=0,description='画像:',continuous_update=False,layout=widgets.Layout(width='60%'))
            expand=widgets.ToggleButton(value=False,description='拡大表示',icon='search-plus')
            dashboard=widgets.HTML(layout=widgets.Layout(width='100%'))
            status=widgets.HTML()
            def show_frame(change=None):
                i=slider.value
                fig,_=make_plot(data,joint_choice.value,i)
                try:
                    buf=io.BytesIO();fig.savefig(buf,format='png',dpi=120)
                    png=buf.getvalue();(out/'image_and_motion.png').write_bytes(png)
                finally:plt.close(fig)
                for name,image in zip(['front','overhead'],data['frame'](i)):
                    if image is not None:Image.fromarray(image).save(out/(name+'_selected.png'))
                camera_panel(data,i).save(out/'selected_camera_frame.png')
                width='1400px' if expand.value else '900px'
                encoded=base64.b64encode(png).decode('ascii')
                # 標準表示は画面幅に収める。拡大時だけ横スクロールを許可する。
                style=('width:1400px;max-width:none;' if expand.value else 'width:900px;max-width:100%;')
                dashboard.value='<div style="overflow-x:auto"><img alt="Camera views and synchronized robot motion plots" style="'+style+'height:auto" src="data:image/png;base64,'+encoded+'"></div>'
                nearest=int(np.argmin(abs(data['t']-data['it'][i])))
                status.value=html.escape(f"画像 {data['it'][i]:.3f}s / 最近傍制御 {data['t'][nearest]:.3f}s / 差 {abs(data['t'][nearest]-data['it'][i]):.3f}s / 工程 {data['trace'][nearest]['phase']}")
            slider.observe(show_frame,names='value');expand.observe(show_frame,names='value')
            display(widgets.HBox([slider,expand]),status,dashboard);show_frame()
            download=widgets.Button(description='表示画像PNGをダウンロード',icon='download',layout=widgets.Layout(width='250px'))
            feedback=widgets.Output()
            def download_png(_):
                with feedback:
                    clear_output(wait=True)
                    try:
                        from google.colab import files
                        files.download(str(out/'image_and_motion.png'))
                    except ImportError:print('保存済みPNG:',out/'image_and_motion.png')
            download.on_click(download_png);display(download,feedback)
            video_button=widgets.Button(description='カメラ映像MP4を保存')
            video_output=widgets.Output()
            def save_video(_):
                video_button.disabled=True
                with video_output:
                    clear_output(wait=True)
                    try:
                        path=export_camera_video(data,out);print('保存:',path);display(Video(str(path),embed=True,width=800))
                    except Exception as exc:print(type(exc).__name__+': '+str(exc))
                    finally:video_button.disabled=False
            video_button.on_click(save_video)
            extra=widgets.Accordion(children=[widgets.VBox([video_button,video_output])],selected_index=None)
            extra.set_title(0,'補助動画の出力（必要な場合だけ開く）');display(extra)
        except Exception as exc:print(type(exc).__name__+': '+str(exc))
        finally:load_button.disabled=False
load_button.on_click(show_episode)
display(choice,joint_choice,load_button,output)
show_episode(None)


## 8 結果とモデルをDriveへ保存・保存状況を確認

**新規実行は1〜7（必要に応じ7.2）→8、Driveの保存済み記録を使う場合は7.1→7.2→8です。**

新規の実行フォルダはDriveへコピーします。既存バックアップは上書きしません。7.1でDrive上の記録を選んだ場合は、7.2が同じ実行フォルダの`visualization/新しい日時/`へ結果を直接保存するため、このセルは保存先を確認します。重みや元の評価記録を再コピーする必要はありません。

現在の実行フォルダが既にバックアップ済みの場合は、今回作成した可視化フォルダだけを追加保存します。学習・評価の変更を差分バックアップする機能ではありません。

Google Driveへのアクセス時にエラーが出た場合は、認証・接続状態を確認し、このセルを個別に再実行してください。


In [ ]:
from pathlib import Path
import shutil
from google.colab import drive

drive.mount('/content/drive')
drive_root=Path('/content/drive/MyDrive').resolve()
vis=Path(VIS_OUT) if globals().get('VIS_OUT') is not None else None
source=(Path(VIS_RUN) if vis is not None and globals().get('VIS_RUN') is not None
        else Path(RUN) if globals().get('RUN') is not None else None)
if source is None or not source.is_dir():
    raise RuntimeError('保存対象がありません。保存済みデータを使う場合は7.1→7.2の順に実行してください。')
if source.resolve().is_relative_to(drive_root):
    if vis is None or not vis.is_dir() or not any(vis.rglob('*.png')):
        raise RuntimeError('Drive上の可視化画像がまだありません。7.2の読み込みと画像表示を確認してください。')
    print('Driveへ直接保存済み:',vis)
    print('元のモデル・評価記録:',source)
else:
    dst=drive_root/'PAI_results'/source.name
    if not dst.exists():
        shutil.copytree(source,dst)
        print('実行フォルダを保存しました:',dst)
    elif vis is not None and vis.is_dir() and vis.resolve().is_relative_to(source.resolve()):
        target=dst/vis.relative_to(source)
        if target.exists():
            raise RuntimeError('可視化の保存先が既に存在します。上書きせず停止しました: '+str(target))
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copytree(vis,target)
        print('今回の可視化結果を追加保存しました:',target)
        print('既存のモデル・評価記録は上書きしていません。')
    else:
        raise RuntimeError('既存バックアップがあります。上書きせず停止しました: '+str(dst))
